In [0]:
import pathlib
from pyspark.sql.functions import row_number, col, current_timestamp
from pyspark.sql.window import Window
from delta.tables import DeltaTable

df = (
    spark.readStream
    .format("delta")
    .table("f1_lakehouse_dev.bronze.laps")
)
spark.sql("""
        CREATE TABLE IF NOT EXISTS f1_lakehouse_dev.silver.laps (
        date_start TIMESTAMP,
        driver_number INT,
        duration_sector_1 DOUBLE,
        duration_sector_2 DOUBLE,
        duration_sector_3 DOUBLE,
        i1_speed DOUBLE,
        i2_speed DOUBLE,
        is_pit_out_lap BOOLEAN,
        lap_duration DOUBLE,
        lap_number INT,
        meeting_key INT,
        segments_sector_1 STRING,
        segments_sector_2 STRING,
        segments_sector_3 STRING,
        session_key INT,
        _rescued_data STRING,
        _source_file_path STRING,
        _modification_time TIMESTAMP,
        _ingestion_time TIMESTAMP,
        _silver_updated_at TIMESTAMP
    )
          """)

def transform_data_clean(batch_df, batch_id):
    print("transforming batch_id: ", batch_id)
    if batch_df.isEmpty():
        return
    
    specs = Window.partitionBy("session_key", "driver_number", "lap_number").orderBy(col("_ingestion_time").desc())
    df_ranked = batch_df.withColumn("row_rank", row_number().over(specs))

    df_ranked_filtered = df_ranked.filter(col("row_rank")==1).drop("row_rank")

    df_deduped_and_casted = df_ranked.withColumns({
        "date_start": col("date_start").cast("timestamp"),
        "driver_number": col("driver_number").cast("int"),
        "duration_sector_1": col("duration_sector_1").cast("double"),
        "duration_sector_2": col("duration_sector_2").cast("double"),
        "duration_sector_3": col("duration_sector_3").cast("double"),
        "i1_speed": col("i1_speed").cast("int"),
        "i2_speed": col("i2_speed").cast("int"),
        "is_pit_out_lap": col("is_pit_out_lap").cast("boolean"),
        "lap_duration": col("lap_duration").cast("double"),
        "lap_number": col("lap_number").cast("int"),
        "meeting_key": col("meeting_key").cast("int"),
        "session_key": col("session_key").cast("int"),
        "st_speed": col("st_speed").cast("int")
    }).withColumn("_silver_updated_at", current_timestamp())


    target = DeltaTable.forName(batch_df.sparkSession, "f1_lakehouse_dev.silver.laps")
    (
        target.alias("t")
        .merge(
            df_deduped_and_casted.alias("s"),
            "t.session_key = s.session_key AND t.driver_number = s.driver_number AND t.lap_number = s.lap_number"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    df.writeStream
    .foreachBatch(transform_data_clean)
    .option("checkpointLocation", "/Volumes/f1_lakehouse_dev/ops/checkpoints/silver/laps")
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()
